# NRP Hosted LLMs: Free AI for Berkeley Researchers

The **National Research Platform (NRP)** hosts several open-weight LLMs that are free to use for researchers at participating institutions (including UC Berkeley). You can access them via:

- **Chat interfaces**: [Open WebUI](https://nrp-openwebui.nrp-nautilus.io) or [LibreChat](https://librechat.nrp-nautilus.io)
- **OpenAI-compatible API**: `https://ellm.nrp-nautilus.io/v1` (what we'll use here)

This notebook shows how to use the API from Python — the same pattern works with OpenAI, Ollama, vLLM, or any other OpenAI-compatible server.

## 1. Setup

We use the `openai` Python package — it works with any OpenAI-compatible API, not just OpenAI itself.

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI

# NRP's OpenAI-compatible endpoint
# Get your token at: https://nrp.ai (log in > LLM Tokens)
NRP_API_KEY = "your-nrp-token-here"  # <-- Replace with your NRP token

client = OpenAI(
    base_url="https://ellm.nrp-nautilus.io/v1",
    api_key=NRP_API_KEY,
)

## 2. What models are available?

NRP hosts several large open-weight models — many larger than what you could run locally.

In [ ]:
models = client.models.list()

print("Available NRP-hosted models:")
print("-" * 50)
for m in models.data:
    print(f"  {m.id}")

## 3. Chat completion

Let's use the same running example from the workshop: writing a physics abstract. We'll ask one of the large hosted models to generate it.

Notice this is the exact same code pattern you'd use with OpenAI's API — just a different `base_url`.

In [ ]:
response = client.chat.completions.create(
    model="qwen3",  # Qwen 3.5 397B — one of the largest available
    messages=[
        {
            "role": "system",
            "content": "You are an expert academic writer specializing in condensed matter physics.",
        },
        {
            "role": "user",
            "content": (
                "Write an abstract for a paper titled: "
                "'Topological Phase Transitions in Monolayer Jacutingaite (Pt2HgSe3): "
                "A Kane-Mele Analysis with Spin-Orbit and Rashba Corrections'. "
                "The paper investigates the topological phase diagram using an extended "
                "Kane-Mele model, identifies QSH insulator, trivial insulator, and "
                "metallic phases, and shows a tunable gap via biaxial strain (4-6%)."
            ),
        },
    ],
    max_tokens=512,
    temperature=0.7,
)

print(response.choices[0].message.content)

Compare this with what we got from the local 3B model on Ollama — the hosted 397B model produces noticeably richer, more field-appropriate output, because it has more parameters to encode domain knowledge.

But it's also running on someone else's hardware. For sensitive unpublished work, local models (Ollama) or fine-tuned models (next section) give you full control.

## 4. Streaming responses

For longer outputs, streaming lets you see the response as it's generated — same UX as ChatGPT.

In [ ]:
stream = client.chat.completions.create(
    model="qwen3",
    messages=[
        {
            "role": "user",
            "content": (
                "Explain what a Z2 topological invariant is and why it matters "
                "for quantum spin Hall insulators, in 3-4 sentences suitable for "
                "a condensed matter physics paper introduction."
            ),
        },
    ],
    max_tokens=256,
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

print()  # newline at the end

## 5. Using our fine-tuned model

After fine-tuning (see the fine-tuning notebook), we have two ways to use our custom model:

**Option A: Load from HuggingFace** — download the adapter and run inference locally in this notebook.

In [ ]:
# Option A: Load the fine-tuned adapter from HuggingFace
# Uncomment and run after you've uploaded your adapter

# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
#
# base_model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-3.2-3B-Instruct", device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
# model = PeftModel.from_pretrained(base_model, "your-username/physics-abstract-writer")
#
# prompt = (
#     '<|system|>You are an expert academic writer specializing in '
#     'condensed matter physics.<|end|>\n'
#     '<|user|>Write an abstract for a paper about spin-orbit coupling '
#     'effects on topological phase transitions in jacutingaite.<|end|>\n'
#     '<|assistant|>'
# )
#
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

**Option B: Query our vLLM endpoint on NRP** — if we've deployed the model as a service (see the deployment YAMLs), we can query it the same way as any hosted model.

In [ ]:
# Option B: Query our vLLM endpoint on NRP
# Uncomment after deploying the model (see nrp_vllm_deployment.yaml)

# our_client = OpenAI(
#     base_url="http://physics-llm-service:8000/v1",  # cluster-internal URL
#     api_key="not-needed",  # vLLM doesn't require auth by default
# )
#
# response = our_client.chat.completions.create(
#     model="physics-adapter",
#     messages=[
#         {"role": "system", "content": "You are an expert academic writer specializing in condensed matter physics."},
#         {"role": "user", "content": "Write an abstract about topological phase transitions in twisted bilayer graphene."},
#     ],
#     max_tokens=256,
# )
# print(response.choices[0].message.content)

## 6. When to use what

| Approach | Cost | Privacy | Quality | Best for |
|----------|------|---------|---------|----------|
| **NRP hosted LLMs** | Free | Data goes to NRP servers | High (large models) | Quick tasks, exploration |
| **Ollama (local)** | Free | Full privacy | Moderate (limited by your hardware) | Sensitive data, offline work |
| **Fine-tuned on NRP** | Free (GPU time) | Your model, your data | Domain-specific | Repeated domain tasks |
| **HuggingFace Inference** | ~$0.60/hr | Data goes to HF | Depends on model | Sharing with collaborators |
| **Commercial APIs** (OpenAI, Anthropic) | Per-token pricing | Data goes to provider | Highest | When quality matters most |